In [62]:
import numpy as np
import json

In [ ]:
ucc_result_path = "/home/qlila/ResearchCode/QMC_Givens_ansatz_results/UCCSD_results/statevector/N2/wavefunction_analysis/ucc_N2_1.098_30.json"
givens_circuit_path = "/home/qlila/ResearchCode/QMC_Givens_ansatz_results/Givens_results/statevector/N2/wavefunction_analysis/1098.qpy"
with open(ucc_result_path, "r") as f:
    ucc_data = json.load(f)
    ucc = np.array(ucc_data["final_params"])
    ucc_circuit_path = ucc_data["circuit_path"]

In [ ]:
import re
from pytket import Circuit
import warnings
from qiskit import qpy

warnings.filterwarnings(
    "ignore", message="The ParameterVector.*is not fully identical.*"
)


qmc_params_len = {
    "HeH": 3,
    "BeH2": 15,
    "LiH_6_spinorbs": 8,
    "LiH_12_spinorbs": 224,
    "N2": 1420,
}


def pad_params(params, used_params_indices, molecule):
    if used_params_indices is None:
        return params
    return np.array(
        [
            (params[used_params_indices.index(k)] if k in used_params_indices else 0.0)
            for k in range(qmc_params_len[molecule])
        ]
    )


def num_to_bitstr(num, nqubits):
    return bin(num)[2:].zfill(nqubits)


def amplitude_array_to_dict(amplitudes) -> str:
    nqubits = int(np.log2(amplitudes.shape[0]))
    mask = np.abs(amplitudes) > 1e-8
    nonzero_indices = np.where(mask)[0]
    nonzero_amplitudes = amplitudes[mask]
    cleaned_amplitudes = [
        (amp.real if np.abs(amp.imag) < 1e-12 else amp) for amp in nonzero_amplitudes
    ]
    # check they all have the same number of 1s in the keys
    nelectrons = bin(nonzero_indices[0]).count("1")
    for idx in nonzero_indices:
        if bin(idx).count("1") != nelectrons:
            print(
                "Amplitude array contains determinants with different number of electrons."
            )
    return {
        num_to_bitstr(det, nqubits): float(p)
        for det, p in zip(nonzero_indices, cleaned_amplitudes)
    }


def ucc_to_givens_params_transform(
    ucc, ucc_circuit_path, givens_circuit_path, molecule="N2"
):

    # UCC final_params are in the same order used by VQE._bind_pytket_parameters,
    # i.e. sorted(free_symbols, key=_symbol_order_key). We convert them to gate-appearance order.

    def symbol_order_key(symbol):
        name = str(symbol)
        parts = re.split(r"(\d+)", name)
        return tuple((1, int(part)) if part.isdigit() else (0, part) for part in parts)

    with open(ucc_circuit_path, "r") as f:
        ucc_circuit = Circuit.from_dict(json.load(f))

    sorted_symbol_names = [
        str(sym) for sym in sorted(ucc_circuit.free_symbols(), key=symbol_order_key)
    ]

    gate_order_symbol_names = []
    seen_symbols = set()
    for cmd in ucc_circuit.get_commands():
        for sym in cmd.free_symbols():
            name = str(sym)
            if name not in seen_symbols:
                seen_symbols.add(name)
                gate_order_symbol_names.append(name)

    name_to_sorted_index = {name: i for i, name in enumerate(sorted_symbol_names)}
    reorder_indices_sorted_to_gate_order = [
        name_to_sorted_index[name] for name in gate_order_symbol_names
    ]

    ucc_gate_order = ucc[reorder_indices_sorted_to_gate_order]

    with open(givens_circuit_path, "rb") as f:
        qiskit_circuit = qpy.load(f)[0]
    used_params_index_list = [
        int(re.findall(r"\d+", qiskit_circuit.parameters[k].name)[0])
        for k in range(len(qiskit_circuit.parameters))
    ]
    ucc_gate_order_for_givens_circuit = pad_params(
        ucc_gate_order * 2,
        used_params_index_list,
        molecule=molecule,
    )

    return ucc_gate_order_for_givens_circuit


def ucc_to_givens_params_transform_with_sign_correction(
    ucc, ucc_circuit_path, givens_circuit_path, molecule="N2"
):  # Hard coded for N2 1.098 with 30 parameters
    # extract sign arrays.
    ucc_sign_array = np.zeros(30)
    givens_sign_array = np.zeros(30)
    for index in range(30):
        givens_wfn = np.load(
            f"/home/qlila/ResearchCode/QMC_Givens_ansatz_results/Givens_results/statevector/N2/wavefunction_analysis/individual_params/givens_param_{index}.npy"
        )
        givens_dict = amplitude_array_to_dict(givens_wfn)

        # check that there are 2 keys only, and get the value for the key that is NOT "1111100011111000"
        if len(givens_dict) != 2:
            print(
                f"Error: expected 2 determinants in givens dict for index {index}, got {len(givens_dict)}"
            )
        for key in givens_dict:
            if key != "1111100011111000":
                givens_sign_array[index] = np.sign(givens_dict[key])
                givens_det = key
                givens_amp = givens_dict[key]

        ucc_wfn = np.load(
            f"/home/qlila/ResearchCode/QMC_Givens_ansatz_results/UCCSD_results/statevector/N2/wavefunction_analysis/individual_params/ucc_param_{index}.npy"
        )
        ucc_dict = amplitude_array_to_dict(ucc_wfn)
        if len(ucc_dict) != 2:
            print(
                f"Error: expected 2 determinants in ucc dict for index {index}, got {len(ucc_dict)}"
            )
        for key in ucc_dict:
            if key != "1111100011111000":
                ucc_sign_array[index] = np.sign(ucc_dict[key])
                ucc_det = key
                ucc_amp = ucc_dict[key]

        if ucc_det != givens_det:
            print(
                f"Error: expected same non-reference determinant in ucc and givens dict for index {index}, got {ucc_det} and {givens_det}"
            )
        if np.abs(givens_amp) - np.abs(ucc_amp) > 1e-6:
            print(
                f"Error: expected same amplitude for non-reference determinant in ucc and givens dict for index {index}, got {ucc_amp} and {givens_amp}"
            )

    ucc_corrected_sign = np.where(
        ucc_sign_array == givens_sign_array,
        ucc,
        -ucc,
    )
    sign_corrected_ucc_params_for_givens_circuit = ucc_to_givens_params_transform(
        ucc_corrected_sign, ucc_circuit_path, givens_circuit_path, molecule=molecule
    )

    return sign_corrected_ucc_params_for_givens_circuit

In [ ]:
ucc_individual_params_path = "/home/qlila/ResearchCode/QMC_Givens_ansatz_results/UCCSD_results/statevector/N2/wavefunction_analysis/individual_params"
givens_individual_params_path = "/home/qlila/ResearchCode/QMC_Givens_ansatz_results/Givens_results/statevector/N2/wavefunction_analysis/individual_params"


def save_individual_params(ucc_params):
    for i, param in enumerate(ucc_params):
        one_param_array = np.zeros_like(ucc_params)
        one_param_array[i] = param
        np.save(f"{ucc_individual_params_path}/ucc_param_{i}.npy", one_param_array)
        # do a one_param_array for givens params too, for the angle corresponding to the same excitation (not same index)
        givens_one_param_array = ucc_to_givens_params_transform(
            one_param_array, ucc_circuit_path, givens_circuit_path, molecule="N2"
        )
        np.save(
            f"{givens_individual_params_path}/givens_param_{i}.npy",
            givens_one_param_array,
        )


# save_individual_params(ucc)

/home/qlila/anaconda3/envs/givens_env/lib/python3.10/site-packages/qiskit/qpy/interface.py:346: UserWarning: The qiskit version used to generate the provided QPY file, 2.0.0, is newer than the current qiskit version 1.4.2. This may result in an error if the QPY file uses instructions not present in this current qiskit version
  warnings.warn(


In [ ]:
corrected_givens_angles = ucc_to_givens_params_transform_with_sign_correction(
    ucc,
    ucc_circuit_path,
    givens_circuit_path=givens_circuit_path,
)

/home/qlila/anaconda3/envs/givens_env/lib/python3.10/site-packages/qiskit/qpy/interface.py:346: UserWarning: The qiskit version used to generate the provided QPY file, 2.0.0, is newer than the current qiskit version 1.4.2. This may result in an error if the QPY file uses instructions not present in this current qiskit version
  warnings.warn(
